# Policy Demo

Demonstrates the three implemented policies and the underlying mathematical building blocks.

| Policy | Strategy |
|---|---|
| `RandomPolicy` | Picks uniformly at random from remaining candidates |
| `HumanPolicy` | Interactive stdin — skipped in this notebook |
| `EntropyPolicy` | Greedy Shannon-entropy maximisation with Bayesian prior |

**Mathematical building blocks (module-level functions in `policy.py`)**

| Function | Formula |
|---|---|
| `pattern_marginal` | $P(f \mid g) = \sum_{w \in S} P(w) \cdot \mathbf{1}[\varphi(g,w)=f]$ |
| `entropy` | $H(p) = -\sum_k p_k \log_2 p_k$ |
| `bayesian_update` | $P(w \mid f, g) \propto P(w) \cdot \mathbf{1}[\varphi(g,w)=f]$ |

**First run:** `WordleGame.build()` computes the pattern matrix (~3 500 × 3 500) and caches it to `data/`. Subsequent runs load from cache instantly.

In [1]:
import numpy as np
from wordle.game import WordleGame
from wordle.pattern import decode_pattern
from wordle.policy import RandomPolicy, EntropyPolicy, pattern_marginal, entropy, bayesian_update

game = WordleGame.build()
pm   = game.pm
print(game)
print(f"Vocabulary: {len(game.words)} words")

WordleGame(words=3527, max_guesses=6)
Vocabulary: 3527 words


## Step 1 — Pattern marginal  $P(f \mid g)$

For a guess $g$ and a set of candidates $S$ with prior $P(w)$, the marginal over feedback patterns is:

$$P(f \mid g) = \sum_{w \in S} P(w) \cdot \mathbf{1}[\varphi(g,w) = f]$$

With a **uniform prior** every candidate contributes equally, so $P(f|g) = |S_f| / |S|$.  
With a **Zipf-weighted prior** common words contribute more mass.

In [2]:
GUESS = "raise"
gi = pm._dist._word_to_idx[GUESS]

candidates = np.arange(len(game.words))
n = len(candidates)

prior_uniform = np.ones(n) / n

probs_zipf = np.array([pm.distribution.probability(w) for w in game.words])
prior_zipf = probs_zipf / probs_zipf.sum()

marg_uniform = pattern_marginal(gi, candidates, prior_uniform, pm)
marg_zipf    = pattern_marginal(gi, candidates, prior_zipf,    pm)

top = np.argsort(marg_uniform)[::-1][:10]

print(f"Guess: '{GUESS}'  — top-10 patterns by probability\n")
print(f"{'Pattern':<14} {'P(f|g) uniform':<18} {'P(f|g) Zipf':<14}")
print("-" * 48)
for f in top:
    if marg_uniform[f] > 0:
        print(f"{decode_pattern(f):<14} {marg_uniform[f]:.4f}            {marg_zipf[f]:.4f}")

print(f"\nDistinct patterns hit : {(marg_uniform > 0).sum()}")
print(f"Entropy (uniform)     : {entropy(marg_uniform):.4f} bits")
print(f"Entropy (Zipf)        : {entropy(marg_zipf):.4f} bits")

Guess: 'raise'  — top-10 patterns by probability

Pattern        P(f|g) uniform     P(f|g) Zipf   
------------------------------------------------
⬛⬛⬛🟨⬛          0.0553            0.0308
⬛⬛⬛🟨🟨          0.0491            0.0328
⬛⬛⬛⬛⬛          0.0468            0.0582
⬛⬛⬛⬛🟨          0.0456            0.0356
🟨⬛⬛⬛🟨          0.0366            0.0558
⬛⬛🟨⬛⬛          0.0352            0.0253
⬛🟨⬛⬛⬛          0.0337            0.0583
⬛🟩⬛⬛⬛          0.0329            0.0140
🟨🟨⬛⬛⬛          0.0278            0.0154
⬛🟨⬛🟨⬛          0.0272            0.0147

Distinct patterns hit : 156
Entropy (uniform)     : 6.0342 bits
Entropy (Zipf)        : 5.7097 bits


## Step 2 — Bayesian update  $P(w \mid f, g)$

After observing feedback $f$ for guess $g$, the posterior is:

$$P(w \mid f, g) = \frac{P(w) \cdot \mathbf{1}[\varphi(g,w)=f]}{P(f \mid g)}$$

This is just: zero out inconsistent words, renormalise.  
We trace the full update sequence for a game where the target is **drawn from the prior**.

In [3]:
rng    = np.random.default_rng(7)
TARGET = pm.distribution.sample(rng)          # draw target from the prior
policy = EntropyPolicy()

candidates = np.arange(len(game.words))
probs_zipf = np.array([pm.distribution.probability(w) for w in game.words])
prior      = probs_zipf / probs_zipf.sum()

state, _ = game.new_game(word=TARGET)

print(f"Target: '{TARGET}'  (sampled from Zipf prior)\n")
print(f"{'Turn':<5} {'Guess':<8} {'Pattern':<13} {'H(P(·|g))':<12} {'P(f*|g)':<10} {'|S| before→after':<18}")
print("-" * 70)

while not state.done:
    guess    = policy(state, game)
    gi       = pm._dist._word_to_idx[guess]
    n_before = len(candidates)

    marg = pattern_marginal(gi, candidates, prior, pm)
    h    = entropy(marg)

    state, observed_pattern, _ = game.step(state, guess, TARGET)
    p_f = float(marg[observed_pattern])

    candidates, prior = bayesian_update(prior, candidates, gi, observed_pattern, pm)

    arrow = f"{n_before} → {len(candidates)}"
    print(f"{state.guess_count:<5} {guess:<8} {decode_pattern(observed_pattern):<13} "
          f"{h:<12.4f} {p_f:<10.4f} {arrow}")

print()
state.show()

Target: 'sense'  (sampled from Zipf prior)

Turn  Guess    Pattern       H(P(·|g))    P(f*|g)    |S| before→after  
----------------------------------------------------------------------
1     tears    ⬛🟩⬛⬛🟨         5.9606       0.0034     3527 → 15
2     susie    🟩⬛🟨⬛🟩         2.1272       0.4435     15 → 1
3     sense    🟩🟩🟩🟩🟩         -0.0000      1.0000     1 → 1



 T  E  A  R  S   ⬛🟩⬛⬛🟨

 S  U  S  I  E   🟩⬛🟨⬛🟩

 S  E  N  S  E   🟩🟩🟩🟩🟩

Solved in 3 guesses

## Top opening guesses by entropy

Ranks every word as a first guess under both priors.

In [4]:
candidates    = np.arange(len(game.words))
n             = len(candidates)
prior_uniform = np.ones(n) / n
probs_zipf    = np.array([pm.distribution.probability(w) for w in game.words])
prior_zipf    = probs_zipf / probs_zipf.sum()

def all_entropies(prior):
    sub      = pm.matrix[:, candidates].astype(np.int32)
    n_words  = sub.shape[0]
    offsets  = np.arange(n_words)[:, np.newaxis] * 243
    flat_idx = (sub + offsets).ravel()
    marginals = np.bincount(
        flat_idx, weights=np.tile(prior, n_words), minlength=n_words * 243
    ).reshape(n_words, 243)
    p     = marginals
    log_p = np.where(p > 0, np.log2(np.where(p > 0, p, 1.0)), 0.0)
    return -np.sum(p * log_p, axis=1)

H_uniform = all_entropies(prior_uniform)
H_zipf    = all_entropies(prior_zipf)

top_idx = np.argsort(H_uniform)[::-1]

print(f"{'Rank':<5} {'Word':<10} {'H uniform':<12} {'H Zipf':<10}")
print("-" * 40)
for rank, gi in enumerate(top_idx[:10], 1):
    print(f"{rank:<5} {game.words[gi]:<10} {H_uniform[gi]:<12.4f} {H_zipf[gi]:.4f}")


Rank  Word       H uniform    H Zipf    
----------------------------------------
1     aires      6.1487       5.8097
2     aries      6.1327       5.8492
3     rates      6.1315       5.8424
4     tears      6.0923       5.9606
5     tales      6.0898       5.8311
6     lanes      6.0819       5.5759
7     raise      6.0342       5.7097
8     cares      6.0335       5.6268
9     dares      6.0205       5.6159
10    earns      6.0190       5.5781


## Aggregate comparison — sampling from the prior

**Why sample from the prior instead of uniformly?**  
The expected guesses under the policy's own prior is:

$$\mathbb{E}_{w \sim P_0}[\text{guesses}(w)] = \sum_w P_0(w) \cdot \text{guesses}(w)$$

Sampling targets proportionally to $P_0(w)$ gives an unbiased Monte-Carlo estimate of this quantity, whereas uniform sampling gives equal weight to rare and common words.

**Why not 3.4?**  
The 3.42 benchmark is for the original Wordle's curated **2 315-word** answer list of common words.  
Our vocabulary has **3 527 words**, including rarer ones that are harder to resolve quickly.  
A larger, less curated vocabulary → higher average guess count, regardless of prior.

In [5]:
from collections import Counter

N_GAMES = 300

def play(policy, target: str):
    state, _ = game.new_game(word=target)
    while not state.done:
        guess = policy(state, game)
        state, _, _ = game.step(state, guess, target)
    return state

def evaluate(policy, targets):
    return [play(policy, t).guess_count for t in targets]

def summarise(name, counts):
    arr      = np.array(counts)
    failures = (arr > 6).sum()
    dist     = Counter(arr.tolist())
    print(f"\n{name}")
    print(f"  Mean guesses  : {arr.mean():.3f}")
    print(f"  Std           : {arr.std():.3f}")
    print(f"  Failures (>6) : {failures} / {len(counts)} ({100*failures/len(counts):.1f}%)")
    print(f"  Distribution  : ", end="")
    for k in sorted(dist):
        print(f"{k}:{dist[k]}", end="  ")
    print()

# Sample targets from the Zipf prior so common words appear proportionally more
rng_target = np.random.default_rng(0)
targets = [pm.distribution.sample(rng_target) for _ in range(N_GAMES)]

print(f"Sampling targets from Zipf prior  (N={N_GAMES})")
random_counts  = evaluate(RandomPolicy(rng=np.random.default_rng(1)), targets)
entropy_counts = evaluate(EntropyPolicy(), targets)

summarise("RandomPolicy",  random_counts)
summarise("EntropyPolicy", entropy_counts)

print()
print("─" * 50)
print(f"Sampling targets uniformly         (N={N_GAMES})")
rng_u   = np.random.default_rng(0)
sample  = rng_u.choice(len(game.words), size=N_GAMES, replace=True)
targets_uniform = [game.words[i] for i in sample]

entropy_counts_u = evaluate(EntropyPolicy(), targets_uniform)
summarise("EntropyPolicy (uniform targets)", entropy_counts_u)

Sampling targets from Zipf prior  (N=300)

RandomPolicy
  Mean guesses  : 4.123
  Std           : 1.017
  Failures (>6) : 0 / 300 (0.0%)
  Distribution  : 2:14  3:65  4:123  5:66  6:32  

EntropyPolicy
  Mean guesses  : 3.360
  Std           : 0.619
  Failures (>6) : 0 / 300 (0.0%)
  Distribution  : 2:16  3:165  4:116  5:1  6:2  

──────────────────────────────────────────────────
Sampling targets uniformly         (N=300)

EntropyPolicy (uniform targets)
  Mean guesses  : 3.647
  Std           : 0.660
  Failures (>6) : 0 / 300 (0.0%)
  Distribution  : 2:7  3:116  4:153  5:24  
